# Transformers Part I (GPTino)

This notebook is a **from-scratch, decoder-only Transformer** (GPT-style) trained on Tiny Shakespeare.

You’ll build the full pipeline end-to-end:

- Download a raw text dataset
- Tokenize it at the **character level**
- Create training batches for **next-token prediction**
- Implement the main GPT building blocks:
  - token + positional embeddings
  - causal self-attention (with a triangular mask)
  - multi-head attention
  - MLP (feed-forward)
  - residual connections + LayerNorm (pre-norm)
- Train the model and monitor train/val loss

Throughout the notebook, each section includes short explanations plus the code right below it.


## 0) Environment setup

We import the libraries we need (PyTorch + utilities), detect whether a GPU is available, and set a random seed for reproducibility.

The GPU assertion is there because training even a small Transformer is noticeably slower on CPU in Colab.


In [ ]:
# import modules

import requests
import torch
from torch import nn, Tensor
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("device:", device)

assert torch.cuda.is_available(), "⚠️ Please enable GPU in Runtime -> Change runtime type"

torch.manual_seed(0)

## 1) Import Dataset

We'll import the dataset from karpathy's repository. This dataset comprises...

In this section we download the **Tiny Shakespeare** dataset as raw text.

The result is a single Python string `text` containing the entire corpus. We’ll treat it as a continuous stream of characters.


In [ ]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = requests.get(url).text

We’ll quickly print a chunk of the dataset to get a feel for what we’re modeling.

This is useful because a character-level model will learn:

- punctuation and capitalization
- formatting and newlines
- word structure and spelling (emerges from characters)

(If the full print is too long, you can print only the first N characters.)


In [ ]:
print(text)

## 2) Tokenizer



A **character-level tokenizer** is the simplest possible tokenizer:

- the vocabulary is the set of unique characters in the training corpus (letters, spaces, punctuation, `\n`, etc.)
- each character gets an integer id
- encoding turns a string into a list of ids (one per character)
- decoding maps ids back to the original string

No BPE merges, no word splitting — we learn everything directly from characters.


In [ ]:
# -----------------------------------------------------------------------------
# Character-level tokenizer (no BPE, no special tokens)
#
# Goal:
#   - Build a vocabulary of *characters* from the training corpus
#   - Map each character to an integer id (stoi: string-to-int)
#   - Map each integer id back to a character (itos: int-to-string)
#
# Why char-level?
#   - Simplest possible tokenizer for a GPT-style decoder
#   - Vocabulary is small (unique chars in corpus)
#   - The model learns spelling/words/structure purely from characters
# -----------------------------------------------------------------------------

class CharTokenizer:
    # Initialize the tokenizer from the training text (the full corpus).
    def __init__(self, training_text: str):
        # Collect the unique characters present in the training data.
        # sorted(...) gives a deterministic, stable id assignment.
        chars = sorted(set(training_text))

        # stoi: char -> integer id
        self.stoi = {ch: i for i, ch in enumerate(chars)}

        # itos: integer id -> char (inverse mapping)
        self.itos = {i: ch for ch, i in self.stoi.items()}

        # Vocabulary size (number of unique characters)
        self.vocab_size = len(chars)

    # Convenience accessor (useful when building the final linear head).
    def get_vocab_size(self) -> int:
        return self.vocab_size

    # Convert a string into a list of integer token ids (one id per character).
    def encode(self, text: str) -> list[int]:
        # NOTE: This assumes every character in `text` exists in the training vocabulary.
        # If you try to encode characters not seen during training, this will raise KeyError.
        return [self.stoi[ch] for ch in text]

    # Convert token ids back into text.
    def decode(self, ids: list[int]) -> str:
        # In the notebook we often pass a torch.Tensor of ids (e.g. data),
        # so we use i.item() to convert each scalar tensor -> Python int.
        return "".join(self.itos[i.item()] for i in ids)


Next we **build the vocabulary from the full training corpus**, then encode the entire dataset into a single 1D tensor of token ids.

This gives us one long sequence `data` of shape `(N,)` where `N` is the number of characters in the dataset.

Why `torch.long`?
Because `nn.Embedding` expects integer indices (not floats).


In [ ]:
# -----------------------------------------------------------------------------
# Build the tokenizer from the full training corpus and encode the text.
#
# We turn the entire corpus into a 1D stream of token ids:
#   data.shape == (N,)
# where N is the total number of characters.
# -----------------------------------------------------------------------------

# instantiate the tokenizer building the string to integer (stoi) and the integer to string (itos)
tokenizer = CharTokenizer(text)

# number of unique characters in the corpus (this will be the model's vocab size)
vocab_size = tokenizer.get_vocab_size()

# ----------YOUR CODE GOES HERE -----------------#
# encode the text to initialize the data. Make sure is a torch.long tensor
# (nn.Embedding expects integer indices: dtype=torch.long)
data = torch.tensor(tokenizer.encode(text), dtype=torch.long)


Before we train anything, we do a quick sanity check:

> If the tokenizer is correct, decoding the encoded dataset should exactly reconstruct the original text.

This is a great debugging step — if this fails, everything downstream will be garbage.


In [ ]:
# -----------------------------------------------------------------------------
# Sanity check: decoding should reconstruct the original text.
#
# If the tokenizer is correct:
#   decode(encode(text)) == text
# -----------------------------------------------------------------------------

# Check that the decoder works using the data variable. The output from the decoder should be the same as text
decoded_text = tokenizer.decode(data)
print(decoded_text)


Now we split the 1D token stream into **train** and **validation** portions.

Then we define two key training hyperparameters:

- `block_size` (`T`): context length (how many previous tokens the model can see)
- `batch_size` (`B`): how many sequences we process in parallel

A decoder-only LM is trained to predict the *next token* at every position in the block.


In [ ]:
# -----------------------------------------------------------------------------
# Train/validation split + core training hyperparameters.
#
# We split the 1D stream of ids into:
#   - train_data: first 90%
#   - val_data:   last 10%
#
# block_size:
#   - context length (T): how many previous tokens the model sees
# batch_size:
#   - number of sequences per batch (B)
# -----------------------------------------------------------------------------

# Let's set the training/validation split
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

# model parameters
block_size = 128
batch_size = 32


### Creating training batches (next-token prediction)

We sample random contiguous chunks from the token stream.

For each chunk:

- `x` is the input context: `data[p : p+T]`
- `y` is the target:      `data[p+1 : p+T+1]`

So `y` is exactly `x` shifted by one position.

Both end up with shape `(B, T)`.


In [ ]:
# -----------------------------------------------------------------------------
# Mini-batch sampler for a decoder-only language model (next-token prediction).
#
# Given a 1D token stream `data_split`, we sample random starting positions p:
#   x = data_split[p : p+T]
#   y = data_split[p+1 : p+T+1]
#
# So y is "x shifted by one": the model learns to predict the next token.
#
# Shapes:
#   x, y : (B, T)  where B=batch_size and T=block_size
# -----------------------------------------------------------------------------

def get_batch(data_split):
    # Pick random starting indices for each sequence in the batch.
    # We ensure there's enough room for x of length T and y of length T.
    starting_indexes = torch.randint(low=0, high=len(data_split) - block_size - 1, size=(batch_size,))

    # Allocate input and target tensors (token ids).
    x = torch.zeros((batch_size, block_size), dtype=torch.long)
    y = torch.zeros((batch_size, block_size), dtype=torch.long)

    # For each random start position, slice out a contiguous chunk.
    for idx, p in enumerate(starting_indexes):
        # Input sequence (context)
        x[idx, :] = data_split[p: p + block_size]

        # Target sequence (next token for each position in x)
        y[idx, :] = data_split[p + 1: p + block_size + 1]

    return x, y


Let’s quickly verify the batch sampler returns tensors of the expected shapes and dtypes.

At this point, `x_batch` and `y_batch` should be integer token ids with shape `(batch_size, block_size)`.


In [ ]:
# -----------------------------------------------------------------------------
# Quick shape check for the batch sampler.
# -----------------------------------------------------------------------------

x_batch, y_batch = get_batch(train_data)
print("x_batch", x_batch)
print("x_batch.shape", x_batch.shape)
print("y_batch", y_batch)
print("y_batch.shape", y_batch.shape)


## 3) Neural network architecture

Now we implement the GPT-style decoder Transformer.

At a high level, for an input sequence of token ids `x` with shape `(B, T)`:

1. Convert ids into vectors via **token embeddings**
2. Add **positional embeddings** so the model knows token order
3. Apply `N` stacked **decoder blocks**, each made of:
   - causal multi-head self-attention
   - an MLP (feed-forward network)
   - residual connections and LayerNorm (pre-norm)
4. Project the final hidden states to vocabulary logits `(B, T, vocab_size)`

Those logits are used for **next-token prediction** with cross-entropy loss.


### 3.1) Embedding Block

This block produces the initial sequence representation:

- **Token embedding**: looks up a learned vector for each token id.
- **Positional embedding**: adds information about each token's position `0..T-1`.

Output shape: `(B, T, d_model)`.


The **embedding block** converts discrete token ids into continuous vectors:

- token embedding: `id -> ℝ^{d_model}`
- positional embedding: `position -> ℝ^{d_model}`

We add them together to produce an input representation of shape `(B, T, d_model)`.

Positional embeddings are crucial because self-attention alone is permutation-invariant — it doesn’t “know” token order unless we inject position information.


In [ ]:
# -----------------------------------------------------------------------------
# Embedding block: token embedding + positional embedding
#
# Input:
#   input : (B, T) token ids (dtype long)
# Output:
#   (B, T, d_model) continuous vectors
# -----------------------------------------------------------------------------

class DecoderEmbedding(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, block_size: int, device: str):
        super().__init__()
        # Token embedding table: maps token ids -> vectors of size d_model
        self.token_embedding = nn.Embedding(vocab_size, d_model, device=device)

        # Positional embedding table: maps positions 0..block_size-1 -> vectors of size d_model
        self.positional_embedding = nn.Embedding(block_size, d_model, device=device)

    def forward(self, input: Tensor) -> Tensor:
        # input (batch_size, block_size)
        # Unpack sequence length T from the input shape.
        _, T = input.shape

        # Look up token embeddings for every position.
        token_embedding = self.token_embedding(input)  # (batch_size, block_size, d_model)

        # Create position indices [0, 1, ..., T-1] on the correct device.
        pos_indices = torch.arange(T, device=input.device)  # (, block_size)

        # Look up positional embeddings for each position.
        positional_embedding = self.positional_embedding(pos_indices)  # (block_size, d_model)

        # Broadcasting adds (block_size, d_model) to (batch_size, block_size, d_model)
        return token_embedding + positional_embedding  # (batch_size, block_size, d_model)


### 3.2) MultiHeaded Attention

We implement **causal self-attention**:

- Queries, keys, values are computed from the same input sequence.
- Attention scores are masked with a **lower-triangular mask** so token *t* can only attend to `<= t`.

We first implement a single attention **Head**, then combine multiple heads into **MultiHeadAttention**.


A single attention **head** implements *scaled dot-product attention* with a **causal (look-ahead) mask**.

Causality is what makes this a decoder-only model:

- token at position `t` can attend to tokens `≤ t`
- it cannot attend to tokens `> t` (the future)

The mask is a lower-triangular matrix that we apply before softmax.


In [ ]:
# -----------------------------------------------------------------------------
# Single attention head (scaled dot-product attention) with a causal mask.
#
# Input:
#   input : (B, T, d_model)
# Output:
#   (B, T, d_head)
#
# Notes:
# - Wq, Wk, Wv are learned linear projections
# - We apply a lower-triangular mask so positions can't attend to the future
# - We scale by sqrt(d_head) for numerical stability (standard Transformer trick)
# -----------------------------------------------------------------------------

class Head(nn.Module):
    def __init__(self, d_model: int, d_head: int, block_size: int, device=None, dtype=None):
        super().__init__()
        # create Wq, Wk, Wv
        self.d_head = d_head

        # Linear projections for queries, keys, and values.
        self.Wq = nn.Linear(in_features=d_model, out_features=d_head, bias=False, device=device,
                            dtype=dtype)  # d_model, d_head
        self.Wk = nn.Linear(in_features=d_model, out_features=d_head, bias=False, device=device,
                            dtype=dtype)  # d_model, d_head
        self.Wv = nn.Linear(in_features=d_model, out_features=d_head, bias=False, device=device,
                            dtype=dtype)  # d_model, d_head

        # create causal mask
        # register_buffer => stored with the module (moved with .to(device)) but not trained
        self.register_buffer("mask", torch.tril(torch.ones(block_size, block_size, device=device)))

    def forward(self, input: Tensor) -> Tensor:
        # input: (B, T, d_model)
        _, T, _ = input.shape  # (batch_size, T, d_model)

        # Project to Q, K, V for this head.
        Q = self.Wq(input)  # (batch_size, T, d_head)
        K = self.Wk(input)  # (batch_size, T, d_head)
        V = self.Wv(input)  # (batch_size, T, d_head)

        # Raw attention scores: Q K^T -> (B, T, T)
        scores = Q @ K.transpose(-2, -1)  # (batch_size, T, T)

        # Scale by sqrt(d_head) to keep softmax well-behaved.
        attention_scores = scores / (self.d_head ** 0.5)

        # Slice the precomputed mask down to the actual sequence length T.
        mask = self.mask[:T, :T]

        # Apply causal masking: set "future" positions to -inf before softmax.
        masked_attention_scores = attention_scores.masked_fill(mask == 0, float("-inf"))  # (batch_size, T, T)

        # Softmax over the last dim gives attention weights that sum to 1 for each query position.
        softmax_weights = torch.softmax(masked_attention_scores, dim=-1)  # (batch_size, T, T)

        # Weighted sum of values -> (B, T, d_head)
        return softmax_weights @ V  # (batch_size, T, d_head)


**Multi-head attention** runs several attention heads in parallel.

Each head can learn a different “view” of the sequence (e.g., short-range vs long-range dependencies).

We concatenate all head outputs back to `(B, T, d_model)` and apply a final linear projection to mix information across heads.


In [ ]:
# -----------------------------------------------------------------------------
# Multi-head attention: run several heads in parallel and concatenate.
#
# Input:
#   input : (B, T, d_model)
# Output:
#   (B, T, d_model)
#
# Steps:
# 1) Split into n_heads heads, each produces (B, T, d_head) where d_head = d_model / n_heads
# 2) Concatenate along the feature dimension -> (B, T, d_model)
# 3) Project back to d_model with Wout
# -----------------------------------------------------------------------------

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, block_size: int, device=None, dtype=None):
        super().__init__()
        # We require exact split: d_model = n_heads * d_head
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"

        # Build all attention heads.
        heads_list = []
        for i in range(n_heads):
            head = Head(d_model=d_model, d_head=d_model // n_heads, block_size=block_size, device=device, dtype=dtype)
            heads_list.append(head)

        # ModuleList registers submodules properly (so parameters are tracked).
        self.heads = nn.ModuleList(heads_list)

        # Output projection that mixes information across heads.
        self.Wout = nn.Linear(in_features=d_model, out_features=d_model, device=device)

    def forward(self, input: Tensor) -> Tensor:
        # input: (B, T, d_model)
        _, T, _ = input.shape  # (batch_size, T, d_model)

        # Compute outputs for each head (list of (B, T, d_head)).
        head_outputs = [head(input) for head in self.heads]

        # Concatenate head outputs back into (B, T, d_model).
        concat = torch.cat(head_outputs, dim=-1)  # (batch_size, T, d_model)

        # Final linear projection back to d_model.
        return self.Wout(concat)  # (batch_size, T, d_model)


### 3.3) MLP

This is the position-wise feed-forward network (FFN) used in Transformers:

- Expands the channel dimension `d_model -> 4*d_model`
- Applies a nonlinearity (GELU)
- Projects back `4*d_model -> d_model`
- Uses dropout for regularization

It is applied independently at each time step (same weights for every position).


The **MLP / feed-forward network** is applied independently at each sequence position.

In GPT-style models it typically expands the channel dimension (`4× d_model`), applies a non-linearity (often GELU), then projects back down to `d_model`.

Even though attention mixes information *across time*, the MLP mixes information *within the feature dimension*.


In [ ]:
# -----------------------------------------------------------------------------
# Feed-forward network (MLP) used inside each Transformer block.
#
# Input/Output:
#   x : (B, T, d_model)
# -----------------------------------------------------------------------------

class MLP(nn.Module):
    def __init__(self, d_model, device):
        super().__init__()
        # First projection: expand channels
        self.fc1 = nn.Linear(d_model, 4 * d_model, device=device)

        # Activation (GELU is standard in GPT-2 style models)
        self.act = nn.GELU()

        # Second projection: back to d_model
        self.fc2 = nn.Linear(4 * d_model, d_model, device=device)

        # Dropout for regularization
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        # Apply the two-layer MLP with activation in between.
        x = self.fc1(x)
        x = self.act(x)
        x = self.fc2(x)
        x = self.dropout(x)
        return x


### 3.3) Decoder Transformer Block

A single GPT decoder block combines **causal self-attention** and an **MLP**, wrapped with residual connections and LayerNorm.


A **decoder Transformer block** combines the two main sub-layers:

1. Causal multi-head self-attention
2. MLP (feed-forward)

Each sub-layer is wrapped with:

- LayerNorm (pre-norm style)
- residual connection
- dropout (regularization)

This “pre-norm + residual” structure is widely used because it tends to train stably, especially as depth increases.


In [ ]:
# -----------------------------------------------------------------------------
# Decoder Transformer block (GPT-style).
#
# The pattern is:
#   x = x + Dropout(Attention(LN(x)))
#   x = x + Dropout(MLP(LN(x)))
#
# This is a "pre-norm" Transformer (LayerNorm before sub-layer),
# which tends to train more stably.
# -----------------------------------------------------------------------------

class DecoderTransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, block_size, device):
        super().__init__()

        # LayerNorm before attention
        self.ln1 = nn.LayerNorm(d_model, device=device)

        # Causal multi-head self-attention
        self.att = MultiHeadAttention(d_model, n_heads, block_size, device=device)

        # LayerNorm before MLP
        self.ln2 = nn.LayerNorm(d_model, device=device)

        # Feed-forward network
        self.mlp = MLP(d_model, device=device)

        # Dropout applied to residual branches
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        # Attention block + residual connection
        x = x + self.dropout(self.att(self.ln1(x)))

        # MLP block + residual connection
        x = x + self.dropout(self.mlp(self.ln2(x)))
        return x


### 3.4 GPTino Model

This is the full decoder-only model:

- Embedding block (token + positional)
- Stack of `n_layers` decoder blocks
- Final LayerNorm
- Linear language-model head to produce logits over the vocabulary

Output logits shape: `(B, T, vocab_size)`.


Now we assemble the full model (**GPTino**):

- Embedding block: `(B, T) -> (B, T, d_model)`
- Stack of `n_layers` decoder blocks
- Final LayerNorm
- Linear “LM head” to produce logits over the vocabulary: `(B, T, vocab_size)`

These logits are what we feed into cross-entropy loss for next-token prediction.


In [ ]:
# -----------------------------------------------------------------------------
# Full decoder-only Transformer ("GPTino").
#
# Forward pass:
# 1) Embed tokens + positions -> (B, T, d_model)
# 2) Run through N decoder blocks (causal self-attention + MLP)
# 3) Final LayerNorm
# 4) Project to vocabulary logits -> (B, T, vocab_size)
# -----------------------------------------------------------------------------

class GPTino(nn.Module):
    def __init__(self, vocab_size, d_model, block_size, n_heads, n_layers, device):
        super().__init__()

        # Token + positional embedding
        self.embedding = DecoderEmbedding(vocab_size, d_model, block_size, device)

        # Transformer block stack
        self.blocks = nn.ModuleList(
            [DecoderTransformerBlock(d_model, n_heads, block_size, device) for _ in range(n_layers)]
        )

        # Final layer norm (GPT-style)
        self.ln_f = nn.LayerNorm(d_model, device=device)

        # Language model head: maps hidden states to logits for each token in vocab
        self.lm_head = nn.Linear(d_model, vocab_size, device=device)

        # Dropout (not used in this forward, but kept for parity with other modules)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        # x: (B, T) token ids
        x = self.embedding(x)  # (B, T, d_model)

        # Apply stacked decoder blocks
        for block in self.blocks:
            x = block(x)  # (B, T, d_model)

        # Final normalization
        x = self.ln_f(x)  # (B, T, d_model)

        # Output logits for next-token prediction
        return self.lm_head(x)  # (B, T, vocab_size)


## 4) Loss estimation

During training we periodically measure loss on both train and validation splits.

We do this inside `torch.no_grad()` to avoid storing gradients (faster + less memory), and we switch the model to `eval()` to disable dropout.

This gives a cleaner signal of how well the model is learning, and whether it’s overfitting.


In [ ]:
@torch.no_grad()
def estimate_loss(model, eval_iters=100):
    model.eval()
    out = {}
    for label, split in [("train", train_data), ("val", val_data)]:
        losses = []
        for _ in range(eval_iters):
            xb, yb = get_batch(split)
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), yb.view(B * T))
            losses.append(loss.item())
        out[label] = sum(losses) / len(losses)
    model.train()
    return out

## 5) Training loop

We instantiate the model with a chosen configuration (`d_model`, number of heads/layers, context length, etc.) and move it to the selected device.

Then we train with **AdamW** (Adam + decoupled weight decay), which is a standard optimizer choice for Transformers.


In [ ]:
model = GPTino(vocab_size=vocab_size, d_model=256, block_size=block_size, n_heads=4, n_layers=6, device=device).to(device)

In [ ]:
print(model)

### Training step (what happens each iteration)

Each iteration does:

1. Sample a batch `(x, y)` of shape `(B, T)`
2. Forward pass to get logits `(B, T, vocab_size)`
3. Compute cross-entropy loss:
   - we reshape to `(B*T, vocab_size)` vs `(B*T)` targets so PyTorch can compute it in one call
4. Backprop (`loss.backward()`)
5. Optimizer step (`optimizer.step()`)

Every `eval_interval` steps we call `estimate_loss()` to print train/val loss.


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

max_iters = 5000
eval_interval = 500

for step in range(max_iters):
    if step % eval_interval == 0:
        losses = estimate_loss(model)
        print(f"step {step}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
    xb, yb = get_batch(train_data)
    xb, yb = xb.to(device), yb.to(device)

    logits = model(xb)
    B, T, C = logits.shape
    loss = F.cross_entropy(logits.view(B * T, C), yb.view(B * T))
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

## 6) Generative text prediction

After training, we can use the model as a **character-level language generator**.

Generation works autoregressively:

1. Provide an initial prompt (a sequence of token ids)
2. Feed it through the model
3. Take the logits for the **last position**
4. Sample (or select) the next token
5. Append it to the sequence
6. Repeat

Because this is a decoder-only model with a **causal mask**, it naturally supports autoregressive generation.


### The `generate` function

This function implements **autoregressive sampling**.

Key ideas:

- We only feed the model the most recent `block_size` tokens (context window).
- We take the logits from the *last time step*.
- We optionally scale them by `temperature`.
- We sample the next token and append it to the sequence.
- We repeat for `max_new_tokens` steps.

The `@torch.no_grad()` decorator disables gradient tracking (important for inference efficiency).


In [ ]:
@torch.no_grad()
def generate(
    model,
    idx,  # (B, T)
    max_new_tokens: int,
    block_size: int,
    temperature: float = 1.0,
):
    model.eval()

    for _ in range(max_new_tokens):
        idx_cond = idx[:, -block_size:]          # (B, T_cond)
        logits = model(idx_cond)                 # (B, T_cond, V)
        logits = logits[:, -1, :]                # (B, V)
        logits = logits / temperature
        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_id], dim=1)
    return idx

### Providing a prompt

We start generation with a text prompt (e.g., `"YODA: "`).

Steps:

1. Encode the prompt using the tokenizer
2. Convert to a `torch.long` tensor
3. Add batch dimension `(1, T)`
4. Move to the correct device

This becomes the initial context the model will continue from.


In [ ]:
prompt = "YODA: "
start_ids = torch.tensor([tokenizer.encode(prompt)], dtype=torch.long, device=device)

out_ids = generate(model=model, idx=start_ids, max_new_tokens=1000, block_size=128, temperature=1.0)

print(out_ids)

### Decoding the generated tokens

The `generate` function returns token ids.

To convert them back to readable text:

- Select the first batch element (`out_ids[0]`)
- Decode ids back to characters using the tokenizer

Because this is character-level, you may see creative spelling, formatting quirks, or emerging structure.


In [ ]:
generated_text = tokenizer.decode(out_ids[0])
print(generated_text)